In [1]:
!pip -q install transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import evaluate
import joblib

from datasets import  Dataset

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import(
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

### check GPU

In [3]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
else:
  print("Running on GPU")

CUDA Available: True
GPU: Tesla T4


### Unzip CFPB dataset

In [4]:
url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

chunks = pd.read_csv(
    url,
    compression="zip",
    chunksize=100000)


In [5]:
from itertools import islice

df = pd.concat(
    islice(chunks, 10),
    ignore_index=True
)

print(df.shape)

/tmp/ipykernel_1759/728271458.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat(
/tmp/ipykernel_1759/728271458.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat(
/tmp/ipykernel_1759/728271458.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat(
/tmp/ipykernel_1759/728271458.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat(
/tmp/ipykernel_1759/728271458.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat(
/tmp/ipykernel_1759/728271458.py:3: DtypeWarning: Columns (5,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat(
/tmp/ipykernel_1759/728271458.py:3: DtypeWarning: Columns (5) have mixed t

(1000000, 16)


In [6]:
df = df.dropna(subset=["Consumer complaint narrative"])

####Data cleaning

In [7]:


import re

def clean_text(text):
    """Basic cleaning for CFPB narratives: lowercase, strip the XXXX
    redaction tokens, keep letters only, collapse whitespace."""
    text = str(text).lower()
    text = re.sub(r'x{2,}', ' ', text)     # redacted PII like "XXXX"
    text = re.sub(r'[^a-z\s]', ' ', text)  # keep letters only
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_narrative'] = df['Consumer complaint narrative'].apply(clean_text)
df[['Consumer complaint narrative', 'clean_narrative']].head(3)


,Consumer complaint narrative,clean_narrative
0,I pay my bills religious on a monthly basis. C...,i pay my bills religious on a monthly basis ca...
2,I am unaware of the debt with Hunter Warfield ...,i am unaware of the debt with hunter warfield ...
3,I am unaware of the debt with Eastern Account ...,i am unaware of the debt with eastern account ...


In [8]:
#remove rare cases
MIN_SAMPLES = 50

counts = df["Issue"].value_counts()

valid = counts[
    counts >= MIN_SAMPLES
].index

df = df[
    df["Issue"].isin(valid)
].copy()

In [9]:
#encode labels
encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['Issue'])

In [10]:
#tran/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    df["clean_narrative"],

    df["label"],

    test_size=0.2,

    random_state=42,

    stratify=df["label"]

)

###Create HuggingFace DataFrames

In [11]:
train_df = pd.DataFrame({
    "label": y_train,
    "text": X_train
})

test_df = pd.DataFrame({
    "label": y_test,
    "text": X_test
})

In [12]:
#convert to dataset

train_dataset = Dataset.from_pandas(
    train_df
)

test_dataset = Dataset.from_pandas(
    test_df
)

###Load DistilBERT Tokenizer

In [13]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [14]:
#function to tokenize every complaint

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="False",
        truncation=True,
        max_length=256
    )

In [15]:
# #apply to dataset

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)


test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/9132 [00:00<?, ? examples/s]

Map:   0%|          | 0/2284 [00:00<?, ? examples/s]

In [16]:
print(train_dataset.column_names)

['label', 'text', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']


In [17]:
#model only needs the tokenized inputs and labels
train_dataset = train_dataset.remove_columns(["text"])

test_dataset = test_dataset.remove_columns(["text"])

In [18]:
#Rename the label column to what Hugging Face expects
train_dataset = train_dataset.rename_column("label", "labels")

test_dataset = test_dataset.rename_column("label", "labels")

In [19]:
#covert dataset to pytorch format
train_dataset.set_format("torch")

test_dataset.set_format("torch")

In [20]:
#dynamic padding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [21]:
#load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(encoder.classes_)
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
print(model.config)

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9",
    "10": "LABEL_10",
    "11": "LABEL_11",
    "12": "LABEL_12",
    "13": "LABEL_13",
    "14": "LABEL_14",
    "15": "LABEL_15",
    "16": "LABEL_16",
    "17": "LABEL_17",
    "18": "LABEL_18",
    "19": "LABEL_19",
    "20": "LABEL_20",
    "21": "LABEL_21",
    "22": "LABEL_22",
    "23": "LABEL_23",
    "24": "LABEL_24",
    "25": "LABEL_25",
    "26": "LABEL_26",
    "27": "LABEL_27",
    "28": "LABEL_28",
    "29": "LABEL_29",
    "30": "LABEL_30",
    "31": "LABEL_31",
    "32": "LABEL_32",
    "33": "LABEL_33",
    "34

### Evaluation

In [23]:
accuracy_metric = evaluate.load("accuracy")

f1_metric = evaluate.load("f1")

In [24]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

##Training

In [25]:
#training argument

training_args = TrainingArguments(
    output_dir="./distilbert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    report_to="none"
)

In [26]:

#creating the trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [27]:
print(train_dataset.column_names)

['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']


##Fine-tuning

In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.351239,2.198273,0.405429,0.283642
2,1.866367,1.906726,0.456217,0.364828
3,1.705125,1.847394,0.470665,0.386425


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1713, training_loss=2.160827959620737, metrics={'train_runtime': 710.1264, 'train_samples_per_second': 38.579, 'train_steps_per_second': 2.412, 'total_flos': 1815231352333536.0, 'train_loss': 2.160827959620737, 'epoch': 3.0})

In [29]:
trainer.save_model("./distilbert_issue_model")

tokenizer.save_pretrained("./distilbert_issue_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./distilbert_issue_model/tokenizer_config.json',
 './distilbert_issue_model/tokenizer.json')

In [45]:
!zip -r distilbert_issue_model.zip distilbert_issue_model


  adding: distilbert_issue_model/ (stored 0%)
  adding: distilbert_issue_model/config.json (deflated 69%)
  adding: distilbert_issue_model/training_args.bin (deflated 53%)
  adding: distilbert_issue_model/tokenizer.json (deflated 71%)
  adding: distilbert_issue_model/model.safetensors (deflated 8%)
  adding: distilbert_issue_model/tokenizer_config.json (deflated 43%)


In [30]:
import joblib

joblib.dump(
    encoder,
    "./distilbert_label_encoder.pkl"
)

['./distilbert_label_encoder.pkl']

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [32]:
#prediction function
def predict_issue(text):

    # Tokenize the complaint
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # Move tensors to GPU (or CPU)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Disable gradient calculation
    with torch.no_grad():
        outputs = model(**inputs)

    # Predicted class index
    predicted_class = torch.argmax(outputs.logits, dim=1).item()

    # Confidence score
    probabilities = torch.softmax(outputs.logits, dim=1)

    confidence = probabilities.max().item()

    # Convert index back to Issue name
    issue = encoder.inverse_transform([predicted_class])[0]

    return {
        "issue": issue,
        "confidence": round(confidence, 4)
    }

In [33]:
predict_issue(
    "My credit card has been charged twice and customer service refuses to help."
)

{'issue': 'Problem with a purchase shown on your statement',
 'confidence': 0.3981}

#Sentiment model using pretrainined model <br>
Adopting Cardiff NLP's model

In [35]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [36]:
#save the pretrained
sentiment_pipeline.model.save_pretrained(
    "./roberta_sentiment"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [37]:
#save the tokenizer
sentiment_pipeline.tokenizer.save_pretrained(
    "./roberta_sentiment"
)

('./roberta_sentiment/tokenizer_config.json',
 './roberta_sentiment/tokenizer.json')

In [46]:
!zip -r roberta_sentiment.zip roberta_sentiment

  adding: roberta_sentiment/ (stored 0%)
  adding: roberta_sentiment/config.json (deflated 53%)
  adding: roberta_sentiment/tokenizer.json (deflated 82%)
  adding: roberta_sentiment/model.safetensors (deflated 7%)
  adding: roberta_sentiment/tokenizer_config.json (deflated 50%)


In [38]:
text = """
My credit card has been charged twice and customer support refuses to help me.
"""

sentiment_pipeline(text)

[{'label': 'negative', 'score': 0.9259020686149597}]

In [39]:
sentiment_pipeline(
    "Thank you for solving my issue so quickly."
)

[{'label': 'positive', 'score': 0.9013500213623047}]

In [40]:
def predict_sentiment(text):

    result = sentiment_pipeline(text)[0]

    return {

        "sentiment": result["label"],

        "confidence": round(result["score"],4)

    }

In [41]:
predict_sentiment(
    "I hate this company."
)

{'sentiment': 'negative', 'confidence': 0.9364}

###combine with DistilBERT classifier

In [42]:
def triage_complaint(text):

    issue = predict_issue(text)

    sentiment = predict_sentiment(text)

    return {
        "Complaint": text,
        "Predicted Issue": issue["issue"],
        "Issue Confidence": issue["confidence"],
        "Sentiment": sentiment["sentiment"],
        "Sentiment Confidence": sentiment["confidence"]
    }

In [43]:
triage_complaint(
    """
    My account has been charged twice and nobody is helping me.
    """
)

{'Complaint': '\n    My account has been charged twice and nobody is helping me.\n    ',
 'Predicted Issue': 'Closing an account',
 'Issue Confidence': 0.1336,
 'Sentiment': 'negative',
 'Sentiment Confidence': 0.9188}

In [44]:
#keep
# issue_model = AutoModelForSequenceClassification.from_pretrained(
#     "models/distilbert_issue"
# )

# sentiment_pipeline = pipeline(
#     "sentiment-analysis",
#     model="models/roberta_sentiment",
#     tokenizer="models/roberta_sentiment"
# )